# Airplane Detection with Faster R-CNN

Run this notebook from top to bottom in Google Colab. It expects this project at `MyDrive/Airplane_Detection_FasterRCNN/` and the dataset at `MyDrive/Airplane_Detection_FasterRCNN/dataset/`. Drive authentication is required by Google. Training, checkpoints, plots, and predictions are written to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT_DIR = Path('/content/drive/MyDrive/Airplane_Detection_FasterRCNN')
DATASET_DIR = PROJECT_DIR / 'dataset'
assert PROJECT_DIR.is_dir(), f'Project folder not found: {PROJECT_DIR}'
assert (PROJECT_DIR / 'src' / 'train.py').is_file(), 'Project source files are missing from the Drive project folder.'
assert (DATASET_DIR / 'annotations.csv').is_file(), f'Annotations not found: {DATASET_DIR / "annotations.csv"}'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)


## Install all project dependencies

In [ ]:
# Install a mutually compatible PyTorch/TorchVision pair and all project libraries.
%pip install -q -r requirements.txt


## Verify the dataset before training

In [ ]:
import ast
import pandas as pd
from PIL import Image
ann = pd.read_csv(DATASET_DIR / 'annotations.csv')
required = {'image_id', 'geometry', 'class'}
assert required.issubset(ann.columns), f'Expected columns {required}; found {set(ann.columns)}'
assert ann[list(required)].notna().all().all(), 'Missing values found in required annotation fields.'
image_dir = DATASET_DIR / 'images'
image_files = {p.name: p for p in image_dir.iterdir() if p.is_file()}
missing = sorted(set(ann.image_id.astype(str)) - set(image_files))
assert not missing, f'{len(missing)} annotated image files are missing; examples: {missing[:5]}'
bad = []
for name, path in image_files.items():
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        bad.append((name, str(exc)))
assert not bad, f'Unreadable image files: {bad[:5]}'
invalid = []
for row_num, geometry in enumerate(ann.geometry, start=2):
    try:
        points = ast.literal_eval(str(geometry))
        assert len(points) >= 4 and all(len(pt) == 2 for pt in points)
    except Exception as exc:
        invalid.append((row_num, str(exc)))
assert not invalid, f'Invalid polygon annotations: {invalid[:5]}'
print(f'Verified {len(image_files)} readable images and {len(ann)} annotation rows.')
print('CSV columns:', ann.columns.tolist())
print('Image classes / annotation counts:\n', ann['class'].value_counts().to_string())
extra_dir = DATASET_DIR / 'extras'
extra_images = sorted(p for p in extra_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})
assert len(extra_images) >= 5, f'At least five sample images are required; found {len(extra_images)}.'
print(f'Unannotated inference examples available: {len(extra_images)}')


## Visualize a labeled training sample

In [ ]:
import matplotlib.pyplot as plt
from src.dataset import AircraftDataset
from src.transforms import get_transforms
sample_ds = AircraftDataset(DATASET_DIR, transforms=get_transforms(False))
image, target = sample_ds[0]
fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(image.permute(1,2,0)); ax.axis('off')
for box, label in zip(target['boxes'], target['labels']):
    x1,y1,x2,y2 = box.tolist(); ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color='lime',lw=2))
    ax.text(x1,y1,str(sample_ds.classes[int(label)-1]),color='black',bbox={'facecolor':'lime','alpha':.8})
plt.show()


## Train Faster R-CNN
The default run uses 20 epochs and batch size 1 to fit the 2560×2560 images on common Colab GPUs. Checkpoints and loss plots are written directly into the Drive project folder.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 1
!python -m src.train --dataset "$DATASET_DIR" --epochs $EPOCHS --batch-size $BATCH_SIZE --workers 2 --checkpoints "$PROJECT_DIR/checkpoints" --output "$PROJECT_DIR/outputs"


In [ ]:
import shutil
trained_checkpoint = PROJECT_DIR / 'checkpoints' / 'best.pth'
assert trained_checkpoint.exists(), 'Training did not produce checkpoints/best.pth.'
best_model_path = PROJECT_DIR / 'best_model.pth'
shutil.copy2(trained_checkpoint, best_model_path)
print('Best model saved to Google Drive:', best_model_path)


## Evaluate on the held-out validation split

In [ ]:
!python -m src.evaluate --dataset "$DATASET_DIR" --checkpoint "$PROJECT_DIR/best_model.pth" --batch-size 1 --workers 2


## Run inference on at least five unannotated sample images

In [ ]:
sample_output_dir = PROJECT_DIR / 'outputs' / 'predictions'
!python -m src.inference "$DATASET_DIR/extras" --checkpoint "$PROJECT_DIR/best_model.pth" --threshold 0.3 --output "$sample_output_dir"


## Display predicted bounding boxes

In [ ]:
from IPython.display import display
prediction_files = sorted(sample_output_dir.glob('*_prediction.jpg'))
assert len(prediction_files) >= 5, f'Expected at least 5 prediction images, found {len(prediction_files)}.'
for prediction_path in prediction_files[:5]:
    print(prediction_path.name)
    display(Image.open(prediction_path))


## Display training loss graph

In [ ]:
loss_plot = PROJECT_DIR / 'outputs' / 'loss.png'
assert loss_plot.exists(), 'Training loss graph was not produced.'
display(Image.open(loss_plot))


## Model artifact

In [ ]:
print(f'Trained model: {best_model_path}')
print(f'Validation metrics were printed above. Prediction images: {sample_output_dir}')
